# Day 065 — Exercise 3: Readiness Checker

Before deploying, run a **readiness check**: a structured set of assertions that verify the app behaves correctly. If any check fails, abort the deploy. This is the principle behind Kubernetes readinessProbes and Render's health checks — automated gates that prevent a broken build reaching production.

Our readiness checker verifies four properties:

| Check | Verifies |
|-------|----------|
| `health_ok` | `/health` returns 200 with `status='ok'` |
| `templates_exist` | `/templates` returns a non-empty list |
| `validation_works` | Empty prompt → 422 (Pydantic validation active) |
| `rate_limit_works` | Valid prompt → 200 (app actually works) |

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# Pre-built apps for checks
def _full_app():
    """Complete writing-assistant-style app."""
    app = FastAPI()
    class _R(BaseModel):
        prompt:  str = Field(min_length=1)
        user_id: str = Field(min_length=1)
    @app.get("/health")
    def h(): return {"status": "ok"}
    @app.get("/templates")
    def t(): return {"templates": ["email", "tweet"]}
    @app.post("/generate")
    def g(req: _R): return {"content_id": "cid", "content": req.prompt.upper(),
                             "user_id": req.user_id}
    return app

def _bare_app():
    """Minimal app — only health, no generate/templates."""
    app = FastAPI()
    @app.get("/health")
    def h(): return {"status": "ok"}
    return app


## Task

Implement `check_readiness(client) -> dict`:

- Run each check in a try/except (exception = failed)
- Collect `{name, passed: bool, detail: str}` per check
- `ready = all(c['passed'] for c in checks)`
- Return `{ready: bool, checks: list[...]}`

## Your Implementation

In [ ]:
def check_readiness(client) -> dict:
    """Run pre-deploy readiness checks against a TestClient.

    Checks performed:
        health_ok        GET /health → 200 with status='ok'
        templates_exist  GET /templates → 200 with non-empty 'templates' list
        validation_works POST /generate with empty prompt → 422
        rate_limit_works POST /generate with valid body → 200 (not 429)

    Returns:
        {
          'ready':  bool,          # True only if ALL checks pass
          'checks': list[{name: str, passed: bool, detail: str}],
        }
    """
    # TODO: run each check, collect {name, passed, detail}, compute ready
    raise NotImplementedError


In [ ]:
def check_readiness(client) -> dict:
    checks = []

    def run(name, fn):
        try:
            passed, detail = fn()
        except Exception as e:
            passed, detail = False, str(e)
        checks.append({"name": name, "passed": passed, "detail": detail})

    def _health():
        r = client.get("/health")
        ok = r.status_code == 200 and r.json().get("status") == "ok"
        return ok, f"status={r.status_code}"

    def _templates():
        r = client.get("/templates")
        if r.status_code != 200:
            return False, f"status={r.status_code}"
        tmpl = r.json().get("templates", [])
        ok = isinstance(tmpl, list) and len(tmpl) > 0
        return ok, f"{len(tmpl)} templates found"

    def _validation():
        r = client.post("/generate", json={"prompt": "", "user_id": "x"})
        ok = r.status_code == 422
        return ok, f"status={r.status_code} (want 422)"

    def _generate():
        r = client.post("/generate", json={"prompt": "hi", "user_id": "x"})
        ok = r.status_code == 200
        return ok, f"status={r.status_code} (want 200)"

    run("health_ok", _health)
    run("templates_exist", _templates)
    run("validation_works", _validation)
    run("rate_limit_works", _generate)

    return {"ready": all(c["passed"] for c in checks), "checks": checks}


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # Full app → ready
    full_client = TestClient(_full_app(), raise_server_exceptions=False)
    r_full = check_readiness(full_client)
    assert isinstance(r_full, dict) and "ready" in r_full and "checks" in r_full
    score += 1; print("\u2705 returns dict with 'ready' and 'checks' keys")

    assert r_full["ready"] is True, f"expected ready=True: {r_full['checks']}"
    score += 1; print("\u2705 full app → ready=True")

    assert isinstance(r_full["checks"], list) and len(r_full["checks"]) >= 3
    score += 1; print("\u2705 checks list has at least 3 entries")

    each_ok = all(isinstance(c, dict) and
                  all(k in c for k in ("name", "passed", "detail"))
                  for c in r_full["checks"])
    assert each_ok, "each check must have name, passed, detail"
    score += 1; print("\u2705 each check has name, passed, detail")

    # Bare app → not ready
    bare_client = TestClient(_bare_app(), raise_server_exceptions=False)
    r_bare = check_readiness(bare_client)
    assert r_bare["ready"] is False, f"expected ready=False: {r_bare['checks']}"
    score += 1; print("\u2705 bare app (missing routes) → ready=False")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def check_readiness(client) -> dict:
    checks = []

    def run(name, fn):
        try:
            passed, detail = fn()
        except Exception as e:
            passed, detail = False, str(e)
        checks.append({"name": name, "passed": passed, "detail": detail})

    def _health():
        r = client.get("/health")
        ok = r.status_code == 200 and r.json().get("status") == "ok"
        return ok, f"status={r.status_code}"

    def _templates():
        r = client.get("/templates")
        if r.status_code != 200:
            return False, f"status={r.status_code}"
        tmpl = r.json().get("templates", [])
        ok = isinstance(tmpl, list) and len(tmpl) > 0
        return ok, f"{len(tmpl)} templates found"

    def _validation():
        r = client.post("/generate", json={"prompt": "", "user_id": "x"})
        ok = r.status_code == 422
        return ok, f"status={r.status_code} (want 422)"

    def _generate():
        r = client.post("/generate", json={"prompt": "hi", "user_id": "x"})
        ok = r.status_code == 200
        return ok, f"status={r.status_code} (want 200)"

    run("health_ok", _health)
    run("templates_exist", _templates)
    run("validation_works", _validation)
    run("rate_limit_works", _generate)

    return {"ready": all(c["passed"] for c in checks), "checks": checks}
```

**The inner `run` helper** eliminates the try/except repetition — write each check as a plain function that returns `(passed, detail)`, then wrap it once. `all(c['passed'] for c in checks)` short-circuits on the first failure — but here we collect all checks first so the report shows everything that failed, not just the first.

</details>